# pyjmri quickstart — interactive

Same flow as the README quickstart, split into cells so the `Client` stays open across the whole session. Run the cells top-to-bottom once, then re-run the **flip** cell as often as you like — it commands the opposite of the current state each time. When you're done, run the **close** cell (or restart the kernel).

In [ ]:
import logging
from pyjmri import Client, TurnoutState

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,                      # use DEBUG to see WebSocket traffic
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",                   # omit to log to stderr instead
    force=True,                              # re-run-safe in Jupyter
)

## Open the client (run once)

`Client.__aenter__` opens the HTTP and WebSocket connections; the returned object stays bound to `jmri` for the rest of the notebook. Re-running this cell on an already-open client raises `RuntimeError` — close it first if you need to reconnect.

In [ ]:
jmri = await Client().__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors")

## Pick a turnout

Grab the first turnout in the layout. To target a specific one instead, swap in `layout.turnouts.by_system_name("NT100")` or `layout.turnouts.by_user_name("North Yard Lead")`.

In [ ]:
turnout = next(iter(layout.turnouts.values()))
print(f"name={turnout.name} user_name={turnout.user_name} initial state={turnout.state.name}")

## Flip it (re-run repeatedly)

Each run commands the opposite of whatever JMRI is currently reporting, so re-running this cell toggles the turnout back and forth.

In [ ]:
target = TurnoutState.THROWN if turnout.state is TurnoutState.CLOSED else TurnoutState.CLOSED
await turnout.set_state(target)
print(f"final state={target.name}")

## Close the client

Run this when you're done to shut down the WebSocket and background tasks cleanly. Kernel restart also cleans up if you forget.

In [ ]:
await jmri.__aexit__(None, None, None)